In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\user\my_snippet\DataSets\Titanic.csv')


In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [3]:
#1 number of missing values
missing_values = df.isnull().sum()
embark = df.groupby('Embarked')
missing_values[missing_values>0]
#2 embar
from_each_port = embark['PassengerId'].count()
from_each_port

#3 average age
#4 highest surving pclass
pclass = df.groupby('Pclass')
pct_survived = (pclass['Survived'].sum()/pclass['PassengerId'].count())*100
pct_survived
#5 embarked from each port

Pclass
1    46.728972
2    32.258065
3    33.027523
dtype: float64

In [4]:
#6 all survived female
survived_female = df[(df['Sex'] == 'female') & (df['Survived'] == 1)]
df.groupby('Sex')['Survived'].sum()
#7 older than 60 years
#8 fare greater than 50
#9 traveled alone -> SibSp(siblings and spouse) ==0 and parch(Parent and children)==0
#10 name contains "Mr."
contains_mr = df[df["Name"].str.contains('Mr.')]
contains_mr.shape

(312, 12)

In [5]:
#11 mean fare each passenger class
#12 avg. age
#13 survivors from each embarkment point
#14 max fair in each class 
max_fare = df.groupby('Pclass')['Fare'].agg(['max'])
#15 survival rate by gender
gb_sex = df.groupby('Sex')
pct_survived = (gb_sex['Survived'].sum()/gb_sex['PassengerId'].count())*100
pct_survived


Sex
female    100.0
male        0.0
dtype: float64

In [6]:
#16 newCol
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
#17 newCol
df['IsAlone'] = ((df['SibSp'] == 0) & (df['Parch'] == 0)).astype(int) 
df['IsAlone'] = np.where((df['FamilySize']) == 1,1,0)#np.where(condition,if_true,if_false)
df.head()

#18 extract titles
df['Title'] = np.where(df['Name'].str.contains('Mr'), 'Mr',
                np.where(df['Name'].str.contains('Miss'), 'Miss',
                np.where(df['Name'].str.contains('Mrs'), 'Mrs', '-')))
#19 child,adult,senior
df['AgeGroup'] = np.where(df['Age']<18,'Child',
                np.where(df['Age'].between(18,60),'Adult',
                np.where(df['Age']>60,'Senior','-')))
#20 fare/member
df['Fare/Member'] = df['Fare']/df['FamilySize']

In [7]:
#21 highest fare payer
df.sort_values('Fare',ascending=False).head(10)
#22 by age and fare
df.sort_values(['Fare','Age'],ascending=False).head(10)
#23 youngest survivor passenger
younger = df['Age'].quantile(0.05)
df[(df['Age']<younger) & (df['Survived'] == 1)].sort_values('Age').head(2)

#24 Rank passengers based on fare within each class.
df['FareRank'] = df.groupby('Pclass')['Fare'].rank(ascending=False)
df.head()
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Cabin'] = df['Cabin'].fillna('Unknown')
df['Embarked'].dropna()#permanent -> df = df.dropna(subset=['Embarked'])
pct_survived = (pclass['Survived'].sum()/pclass['PassengerId'].count())*100
pct_survived# pct_survived = pclass['Survived'].mean() * 100 faster version


Pclass
1    46.728972
2    32.258065
3    33.027523
dtype: float64

In [8]:
#30 average fare by pivot_table
df.pivot_table(index='Sex',columns='Pclass',values='Fare',aggfunc='mean')

#31 create a pivot table 
df.pivot_table(index='Sex',columns='Embarked',values='Survived',aggfunc='sum')

#32 identify duplicate ticket number
df['Ticket'].value_counts().sort_values(ascending=False)

#33 families traveling together
df['Surname'] = df["Name"].str.split(',').str[0]
df.groupby('Surname')['FamilySize'].sum()

#34 calculate z score z=x-~x/std
mean_fare = df['Fare'].mean()
std_fare = df['Fare'].std()
z = (df['Fare']-mean_fare)/std_fare
df['ZScore'] = z
#detect outliers
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)
IQR = q3 - q1

outliers = df[(df['Fare'] < q1 -1.5*IQR) | (df['Fare'] > q3+1.5*IQR)]
outliers.shape

(55, 20)

In [ ]:
df.groupby('AgeGroup')['Survived'].mean()*100
df.groupby('Cabin')['Survived'].mean()*100
df['WithCabin'] = (~df['Cabin'].str.contains('Unknown')).astype(int)
df.groupby('WithCabin')['Survived'].mean()*100
####################important###################
bins = np.arange(0,df['Age'].max()+10,10)
df["ages"] = pd.cut(df['Age'],bins=bins)
df.groupby('ages')['Survived'].mean()*100

ages
(0.0, 10.0]      45.454545
(10.0, 20.0]     46.808511
(20.0, 30.0]     34.351145
(30.0, 40.0]     31.914894
(40.0, 50.0]     32.608696
(50.0, 60.0]     55.000000
(60.0, 70.0]     30.000000
(70.0, 80.0]    100.000000
Name: Survived, dtype: float64

In [24]:
df.groupby('Survived')['Fare'].mean()
wealthiest_port = df.groupby('Embarked')['Fare'].sum().idxmax()
wealthiest_port

'S'

In [38]:
total_passenger = df['PassengerId'].nunique()
dead = total_passenger - df['Survived'].sum()
survivors = df['Survived'].sum()
avg_fare = df['Fare'].mean()
avg_age = df['Age'].mean()
#summary table
df.pivot_table(index='Pclass',values=['Fare','Age','Survived'],aggfunc=['mean','mean','sum'])
summary = df.pivot_table(
    index='Pclass',
    values=['PassengerId', 'Survived', 'Fare', 'Age'],
    aggfunc={
        'PassengerId': 'count',
        'Survived': 'sum',
        'Fare': 'mean',
        'Age': 'mean'
    }
)
df.groupby('Pclass').agg(
    total_passengers=('PassengerId', 'count'),
    survivors=('Survived', 'sum'),
    avg_fare=('Fare', 'mean'),
    avg_age=('Age', 'mean')
)
summary

,Age,Fare,PassengerId,Survived
Pclass,,,,
1,40.022928,94.280297,107,50
2,28.857881,22.202104,93,30
3,26.090397,12.459678,218,72
